### Customer Support Resolution Agent (MCP)

#### Problem it addresses

Customers expect fast, accurate answers and a single thread they can return to. In real support queues, issues are lost when conversations are not tied to tickets, policy answers drift from the official knowledge base, and follow-up messages (“still broken on ticket TKT-…”) need full context from prior agent replies. This notebook models that situation with SQLite-backed tickets, per-ticket message threads, and a small internal KB—so the agent must **read and write systems of record**, not improvise from memory.

#### Principle

Keep **business rules and data** in ordinary Python and SQL (`support.py`), and expose them to the model through **MCP tools** with a stable contract. The LLM plans language and next steps; durable state (status, notes, thread rows) lives in the database. That separation mirrors how production stacks integrate CRMs, wikis, and ticketing APIs: the assistant stays thin; the tools enforce structure.

#### Approach

1. **Sanity-check the domain layer** by calling ticket and KB functions directly (no MCP), matching what the server will wrap.  
2. **Run the same surface area as a stdio MCP server** (`uv run support_server.py`) and list tools from the notebook.  
3. **Bind the server to an OpenAI Agents `Agent`**, run `Runner.run` under `trace`, and encode a ticket lifecycle in instructions (create → thread messages → escalate/close/reopen). Inbound text that contains a ticket id (`TKT-` + 10 hex) can be **pre-appended** to the thread so follow-ups persist even if the model omits a tool call.  
4. **Exercise follow-ups** on a seeded resolved ticket and inspect rows with `get_ticket_messages`.

#### Where this pattern applies

Internal IT helpdesks, B2B SaaS support, e-commerce post-sale care—anywhere you need grounded answers, auditable actions, and the option to swap backends (different MCP servers or tools) without rewriting the whole agent.

#### What you run in this notebook

Loads environment variables, resolves `AGENT_DIR` (folder containing `support_server.py`), uses **`support.py`** for direct ticket/KB calls, then starts **`MCPServerStdio`**, lists MCP tools, and runs **`handle_customer_request`** (`Runner.run` + Markdown display). Later cells demo threaded follow-ups and **`get_ticket_messages`** for inspection.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown


def resolve_agent_dir() -> Path:
    here = Path.cwd()
    if (here / "support_server.py").exists():
        return here
    return here

model = "gpt-4.1-mini"

AGENT_DIR = resolve_agent_dir()
load_dotenv(override=True)

AGENT_DIR

### First, explore the support system directly (no MCP yet)

In [ ]:
from support import create_ticket, get_ticket, escalate_ticket, close_ticket

t = create_ticket("Jamie Rivera", "Reset link expires before I can click it", "medium")
print("Created", t.ticket_id, t.status)
print(escalate_ticket(t.ticket_id, "Possible email delay; customer tried twice."))
print(close_ticket(t.ticket_id, "Walked customer through reset; login confirmed."))
get_ticket(t.ticket_id)

In [ ]:
from support import search_kb

search_kb("account")

### Now launch it as an MCP server

In [ ]:
support_params = {
    "command": "uv",
    "args": ["run", "support_server.py"],
    "cwd": str(AGENT_DIR),
}

async with MCPServerStdio(
    params=support_params, client_session_timeout_seconds=30
) as server:
    mcp_tools = await server.list_tools()

mcp_tools

In [ ]:
print(mcp_tools)

### Run an agent against the support server

In [ ]:
instructions = """
    You are a customer support agent. You receive requests from customers;
    prioritize, resolve, or escalate when you cannot resolve alone.

    Use ticket tools and search_knowledge_base before giving policy answers. Be concise.

    Ticket and thread workflow:
    - If the customer message contains a ticket id matching TKT- plus 10 hex characters, the runtime may pre-append that inbound text to the thread before you run; call get_ticket_thread first for full context (it should include this turn's customer message). Do not append a duplicate customer row for the same inbound text.
    - For a new issue (no ticket id in the message), after create_ticket you must append_ticket_message with role customer and the customer's current message (verbatim or tight summary).
    - Before close_ticket, call append_ticket_message with role agent and the customer-facing reply to persist for future follow-ups.
    - If the ticket is resolved and the customer needs more help on the same issue, call reopen_ticket with a short reason, then continue (KB, update, escalate, or close). For an unrelated new problem, create_ticket and mention the prior ticket id in the issue text.

    In your final output:
    - Summarize the issue in one short paragraph.
    - Use bullet points for action steps when relevant.
    - Include the ticket id and resolution timing when you close a ticket.
    - End with a clear "Response to customer:" section including the ticket id, professional and concise.

"""

In [ ]:
import re
from support import append_ticket_message

_TICKET_ID_IN_TEXT = re.compile(r"TKT-[0-9a-f]{10}", re.IGNORECASE)


async def handle_customer_request(customer_request: str) -> str:
    """
    Handle customer request and return a response.
    Pre-logs inbound text to the ticket thread when the message cites TKT-xxxxxxxxxx so follow-ups
    persist even if the model skips append_ticket_message.
    """
    print("Handling customer request...")
    print("=" * 20)

    text = customer_request.strip()
    m = _TICKET_ID_IN_TEXT.search(text)
    if m:
        tid = m.group(0).lower()
        try:
            append_ticket_message(tid, "customer", text)
        except ValueError:
            pass

    async with MCPServerStdio(
        params=support_params, client_session_timeout_seconds=30
    ) as mcp_server:
        agent = Agent(
            name="customer_support",
            instructions=instructions,
            model=model,
            mcp_servers=[mcp_server],
        )

        with trace("customer_support_agent"):
            result = await Runner.run(agent, customer_request)
        display(Markdown(result.final_output))


    print("Customer request handled.")



In [ ]:
customer_request = f""" My name James brown, I have trouble resetting my password. 
Can you help me?
"""

await handle_customer_request(customer_request)

### Follow-up demo (ticket thread)

In [ ]:
from support import append_ticket_message, close_ticket, create_ticket, get_ticket_thread

_thread_demo = create_ticket(
    "James Brown",
    "Trouble resetting password",
    "medium",
)
append_ticket_message(
    _thread_demo.ticket_id,
    "customer",
    "I cannot reset my password; please help.",
)
append_ticket_message(
    _thread_demo.ticket_id,
    "agent",
    "Hello James Brown, your password reset issue has been addressed. Use the reset link on the login page; check spam if the email is missing. Ticket ID "
    + _thread_demo.ticket_id
    + ".",
)
close_ticket(
    _thread_demo.ticket_id,
    "Provided KB password reset steps; customer acknowledged.",
)
DEMO_FOLLOWUP_TICKET_ID = _thread_demo.ticket_id
print(DEMO_FOLLOWUP_TICKET_ID)
print(get_ticket_thread(DEMO_FOLLOWUP_TICKET_ID))

In [ ]:
follow_up = f"""Hi, James Brown again. I already spoke with support and they gave me steps for ticket {DEMO_FOLLOWUP_TICKET_ID},
but the reset email never arrives even after checking spam. What should I do next?
"""

await handle_customer_request(follow_up)

In [ ]:
DEMO_FOLLOWUP_TICKET_ID="TKT-fb7bc3563e"

follow_up_updated = f"""Hi, James Brown again. Follow up on ticket {DEMO_FOLLOWUP_TICKET_ID},
Here is my email address: jamesbrown@gmail.com can you confirm that this is the correct email address?
Let me know if there are other reset options that i can use.
"""

await handle_customer_request(follow_up_updated)